# 3단계: 최종 모델 학습과 test 1회 평가

| 순서 | 노트북 | 내용 |
|---|---|---|
| 0 | `ml_0_dataset` | 학습 데이터셋 결정 (705명 vs 원래 445명 출신 → 445명). class weight를 Optuna로 함께 튜닝 (결과 `kfold_logloss`) |
| 1 | `ml_1_main` | class weight 고정으로 학습하고 모델 비교 (결과 `kfold_logloss_balanced`). 6번에서 최종 모델 1개 선택 |
| 2 | `ml_2_test_all_models` | 참고: 0·1의 445명 모델 전체 test 성능 (모델 선택에 쓰지 않음) |
| 3 | `ml_3_final_test` | 최종 모델 1개 재학습과 test 1회 평가 (최종 모델을 고른 뒤 실행) |

**최종 모델을 확정한 뒤에만 실행합니다.** test 89명(SCD 10명)은 모델 선택에 한 번도 쓰지 않았고, 여기서 **한 번만** 평가합니다. 두 번째 평가는 코드에서 막혀 있습니다.

1. 교차검증에서 고른 하이퍼파라미터로 개발 환자 전체를 다시 학습 (재튜닝 없음, `ml_0_dataset`에서 정한 학습 데이터셋: 705명 또는 445명 출신 356명)
2. test 89명 평가: threshold 0.5, 확률 보정 없음
3. 같은 89명에서 비교 기준 계산: Pro-BNP 단독 로지스틱, 진료 지침 기준 LVEF ≤ 35%

## 0. 준비

In [ ]:
import sys
from pathlib import Path

# 저장소 루트를 찾아 import 경로에 추가합니다 (노트북을 어느 폴더에서 열어도 동작).
repo = next(
    p for p in [Path.cwd(), *Path.cwd().parents] if (p / "Members/mingyu/model/ML/kfold").is_dir()
)
sys.path.insert(0, str(repo))

from Members.mingyu.model.ML.kfold import (  # noqa: E402
    Settings,
    evaluate_on_test,
    fit_final,
    list_results,
    load_cohort,
    save_or_check_folds,
)  # noqa: E402

settings = Settings()
cohort = load_cohort(settings)
save_or_check_folds(cohort, settings)  # 처음엔 fold를 저장하고, 이후엔 같은 fold인지 확인

print(
    f"불러온 개발 데이터 {len(cohort.ids)}명 | 이 노트북에서 쓰는 환자: 원래 445명 출신 {cohort.original.sum()}명 "
    f"(SCD {cohort.y[cohort.original].sum()}) | test {len(cohort.test_ids)}명"
)
print("결과 폴더:", settings.result_dir)

## 1. 최종 모델 지정

`ml_1_main` 6번에서 고른 모델 1개를 적고, 선택 근거도 함께 적어 둡니다.


In [ ]:
print("학습된 결과:", list_results(settings))

final_name = None  # 예: Method("Logistic", "MUSIC_SCD").name (445명이면 Method("Logistic", "MUSIC_SCD", "only445").name)
# 선택 근거:
assert final_name, "ml_1_main 6번에서 최종 모델을 확정한 뒤 이름을 적어 주세요"

## 2. 최종 모델 학습 (개발 환자 전체)

모델 파일: `<결과 폴더>/<이름>/final_model.joblib`


In [ ]:
final_model = fit_final(settings, cohort, final_name)
for member in final_model:
    print(
        f"{member.method.model} / {member.method.features}: 학습 {member.n_train}명, 설정 {member.parameters}"
    )

## 3. test 평가 (1회)

결과: `<결과 폴더>/test/<시각>_<이름>/` (`metrics.json`, `test_predictions.csv`)

In [ ]:
evaluate_on_test(settings, cohort, final_name, final_model)